In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
# seting tampilan agar rapi
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

# load data hasil ekstraksi tadi dengan pdfplumber
df = pd.read_csv("../data/processed/ipg_jateng_2025.csv")

print(f"Jumlah Baris: {df.shape[0]}")
print(f"Jumlah Kolom: {df.shape[1]}")
print("\n=== 5 Data Teratas ===")
display(df.head())

Jumlah Baris: 35
Jumlah Kolom: 5

=== 5 Data Teratas ===


,Kabupaten/Kota,IPG,IPM,IPM Laki-laki,IPM Perempuan
0,Cilacap,88.22,73.45,78.25,69.03
1,Banyumas,90.01,75.44,79.34,71.41
2,Purbalingga,94.34,71.71,75.11,70.86
3,Banjarnegara,95.15,70.61,72.54,69.02
4,Kebumen,93.60,73.17,76.62,71.72


In [9]:
# 1. Cek tipe data masing-masin kolom
print("=== Tipe Data ===")
print(df.dtypes)

# 2. Cek jumlah Missing Values (NaN) di setiap kolom
print("\n=== Jumlah Missing Values (NaN) ===")
print(df.isnull().sum())

# 3. Tampilkan baris mana saja yang punya NaN
print("\n=== Baris yang ada NaN-nya ===")
display(df[df.isnull().any(axis=1)])

=== Tipe Data ===
Kabupaten/Kota        str
IPG               float64
IPM               float64
IPM Laki-laki     float64
IPM Perempuan     float64
dtype: object

=== Jumlah Missing Values (NaN) ===
Kabupaten/Kota    0
IPG               0
IPM               2
IPM Laki-laki     4
IPM Perempuan     1
dtype: int64

=== Baris yang ada NaN-nya ===


,Kabupaten/Kota,IPG,IPM,IPM Laki-laki,IPM Perempuan
6,Wonosobo,93.22,71.50,NaN,69.86
7,Magelang,93.31,73.09,NaN,72.00
9,Klaten,96.35,78.68,NaN,80.65
11,Wonogiri,92.56,NaN,77.25,71.50
12,Karanganyar,97.15,NaN,78.67,79.70
34,Provinsi Jawa Tengah,93.73,74.77,NaN,NaN


In [13]:
# 1. salin dataframe untuk jaga-jaga
df_clean = df.copy()

# 2. Perbaiki IPM Laki-laki yang kosong pakai rumus matematika
# Rumus = IPM_Laki = (IPM_Total * 2) - IPM_Perempuan
kondisi_laki_kosong = df_clean['IPM Laki-laki'].isnull() & df_clean['IPM'].notnull() & df_clean['IPM Perempuan'].notnull()

df_clean.loc[kondisi_laki_kosong, 'IPM Laki-laki'] = (
    (df_clean.loc[kondisi_laki_kosong, 'IPM'] * 2) - df_clean.loc[kondisi_laki_kosong, 'IPM Perempuan']
).round(2)

# 3. Cek kembali data kosong
print("Sisa NaN setelah perbaikan:")
print(df_clean.isnull().sum())

# 4. Tampilkan hasilnya
print("\n=== Data Setelah Diperbaiki ===")
display(df_clean.loc[[6, 7, 9, 11, 12]])

Sisa NaN setelah perbaikan:
Kabupaten/Kota    0
IPG               0
IPM               2
IPM Laki-laki     1
IPM Perempuan     1
dtype: int64

=== Data Setelah Diperbaiki ===


,Kabupaten/Kota,IPG,IPM,IPM Laki-laki,IPM Perempuan
6,Wonosobo,93.22,71.50,73.14,69.86
7,Magelang,93.31,73.09,74.18,72.00
9,Klaten,96.35,78.68,76.71,80.65
11,Wonogiri,92.56,NaN,77.25,71.50
12,Karanganyar,97.15,NaN,78.67,79.70


In [24]:
# 1. Isi manual angka Wonogiri (Angka asli IPM Wonogiri di PDF adalah 73,42)
df_clean.loc[df_clean['Kabupaten/Kota'] == 'Wonogiri', 'IPM'] = 73.42

# 2. Hitung ulang IPM Laki-laki untuk Wonogiri
kondisi_wonogiri = df_clean['Kabupaten/Kota'] == 'Wonogiri'
df_clean.loc[kondisi_wonogiri, 'IPM Laki-laki'] = (
    (df_clean.loc[kondisi_wonogiri, 'IPM'] * 2) - df_clean.loc[kondisi_wonogiri, 'IPM Perempuan']
).round(2)

# 3. Hapus baris Provinsi Jawa Tengah karena datanya tidak lengkap dan bukan level kabupaten
df_clean = df_clean[df_clean['Kabupaten/Kota'] != 'Provinsi Jawa Tengah']

# 4. Cek sekali lagi data kosong, memastikan semuanya 0
print("=== Cek Missing Values ===")
print(df_clean.isnull().sum())

# 5. Tampilkan data final
print("\n=== DATA FINAL BERSIH ===")
display(df_clean.head())

=== Cek Missing Values ===
Kabupaten/Kota    0
IPG               0
IPM               1
IPM Laki-laki     0
IPM Perempuan     0
dtype: int64

=== DATA FINAL BERSIH ===


,Kabupaten/Kota,IPG,IPM,IPM Laki-laki,IPM Perempuan
0,Cilacap,88.22,73.45,78.25,69.03
1,Banyumas,90.01,75.44,79.34,71.41
2,Purbalingga,94.34,71.71,75.11,70.86
3,Banjarnegara,95.15,70.61,72.54,69.02
4,Kebumen,93.60,73.17,76.62,71.72


In [29]:
# 1. Cari baris yang IPM-nya masih kosong (NaN)
kondisi_ipm_kosong = df_clean['IPM'].isnull()

# 2. Cetak nama daerahnya agar tahu siapa saja yang tersisa
print("Daerah yang IPM-nya masih kosong:")
print(df_clean.loc[kondisi_ipm_kosong, 'Kabupaten/Kota'].values)

# 3. Isi IPM yang kosong pakai rumus: (IPM Laki-laki + IPM Perempuan) / 2
df_clean.loc[kondisi_ipm_kosong, 'IPM'] = (
    (df_clean.loc[kondisi_ipm_kosong, 'IPM Laki-laki'] + df_clean.loc[kondisi_ipm_kosong, 'IPM Perempuan']) / 2
).round(2)

# 4. Cek Final: Memastikan tidak ada satupun data yang NaN tersisa!
print("\n=== Total Sisa NaN di Seluruh Data ===")
print(df_clean.isnull().sum().sum())

# 5. Tampilkan baris Karanganyar untuk memastikan sudah terisi
print("\n=== Cek Karanganyar ===")
display(df_clean[df_clean['Kabupaten/Kota'] == 'Karanganyar'])

Daerah yang IPM-nya masih kosong:
<StringArray>
[]
Length: 0, dtype: str

=== Total Sisa NaN di Seluruh Data ===
0

=== Cek Karanganyar ===


,Kabupaten/Kota,IPG,IPM,IPM Laki-laki,IPM Perempuan
12,Karanganyar,97.15,79.18,78.67,79.7


In [30]:
# Simpan ke folder data/processed/
output_path = "../data/processed/ipg_jateng_clean.csv"
df_clean.to_csv(output_path, index=False)

print(f"Sukses! Data yang sudah 100% bersih disimpan di: {output_path}")

Sukses! Data yang sudah 100% bersih disimpan di: ../data/processed/ipg_jateng_clean.csv
